# 05 – Load Database
Goal: load the cleaned tables from `data/processed` into the PostgreSQL database `candy_logistics`, whose schema is defined in `sql/01_create_tables.sql` (primary and foreign keys).
Tables are loaded in dependency order, so that every foreign key points to an existing row.

  To reload the data, run `sql/01_create_tables.sql` first (it drops and recreates all tables).

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()
engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

P = "../data/processed/"
factories = pd.read_csv(P + "factories.csv")
products = pd.read_csv(P + "products.csv")
zips = pd.read_csv(P + "zips.csv", dtype={"zip": str})
sales = pd.read_csv(P + "sales.csv", dtype={"postal_code": str}, parse_dates=["order_date", "ship_date"])
road = pd.read_csv(P + "road_distances.csv", dtype={"zip": str})
formulas = pd.read_csv(P + "cost_formulas.csv")

# Normalization: product name and division live in the products table only
sales = sales.drop(columns=["country_region", "division", "product_name"])

tables = [("factories", factories), ("products", products), ("zips", zips),
          ("sales", sales), ("road_distances", road), ("cost_formulas", formulas)]

for name, df in tables:
    df.to_sql(name, engine, if_exists="append", index=False)
    print(f"{name}: {len(df)} rows loaded")

factories: 5 rows loaded
products: 15 rows loaded
zips: 631 rows loaded
sales: 9994 rows loaded
road_distances: 3155 rows loaded
cost_formulas: 3 rows loaded


In [2]:
from sqlalchemy.types import CHAR, VARCHAR

orders = pd.read_csv(P + "orders_scenarios.csv", dtype={"postal_code": str})

orders.to_sql(
    "orders_scenarios", engine, if_exists="replace", index=False,
    dtype={"order_id": VARCHAR(30), "postal_code": CHAR(5), "current_factory": VARCHAR(50),
           "realistic_factory": VARCHAR(50), "optimum_factory": VARCHAR(50)},
)

with engine.begin() as conn:
    conn.execute(text("""
        ALTER TABLE orders_scenarios
            ADD PRIMARY KEY (order_id),
            ADD FOREIGN KEY (postal_code) REFERENCES zips(zip),
            ADD FOREIGN KEY (current_factory) REFERENCES factories(factory),
            ADD FOREIGN KEY (realistic_factory) REFERENCES factories(factory),
            ADD FOREIGN KEY (optimum_factory) REFERENCES factories(factory);
    """))
print(f"orders_scenarios: {len(orders)} rows loaded, keys added")

orders_scenarios: 8389 rows loaded, keys added
